# Experiment 1 - Grover's Search Algorithm
Searches N items in **O(√N)** steps. Quantum speedup over classical O(N) search.

In [ ]:
!pip install qiskit qiskit-aer matplotlib -q

## Imports

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import GroverOperator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

## Part 1— 3 Qubit Search (N = 8, 1 iteration optimal)

### Oracle  marks the target state by flipping its phase

In [ ]:
def make_oracle_3(target):
    circuit = QuantumCircuit(3)
    flipped = target[::-1]

    for position, bit in enumerate(flipped):
        if bit == '0':
            circuit.x(position)

    circuit.h(2)
    circuit.ccx(0, 1, 2)
    circuit.h(2)

    for position, bit in enumerate(flipped):
        if bit == '0':
            circuit.x(position)

    circuit.name = "Oracle"
    return circuit

 Full Grover Circuit  superposition->oracle ->diffuser ->measure

In [ ]:
def run_grover_3(target, shots=1024):
    oracle    = make_oracle_3(target)
    grover_op = GroverOperator(oracle)

    search = QuantumCircuit(3, 3)
    search.h([0, 1, 2])
    search.append(grover_op, [0, 1, 2])
    search.measure([0, 1, 2], [0, 1, 2])

    machine  = AerSimulator()
    compiled = transpile(search, machine)
    result   = machine.run(compiled, shots=shots).result()

    return search, result.get_counts()

### Run and Plot

In [ ]:
target_state     = "100"
circuit_3, data_3 = run_grover_3(target_state)

print(f"Grover Search for |{target_state}>:")
print(data_3)

plot_histogram(data_3, title=f"Grover 3-Qubit - Target |{target_state}>", figsize=(7, 4))
plt.tight_layout()
plt.show()

### Draw Circuit

In [ ]:
circuit_3.draw('mpl')

---
## Part 2-4 Qubit Search (N = 16, 2 iterations optimal, multiple targets)

### Oracle for 4 Qubits with Multiple Targets

In [ ]:
def make_oracle_4(targets):
    circuit = QuantumCircuit(4)

    for target in targets:
        flipped = target[::-1]

        for position, bit in enumerate(flipped):
            if bit == '0':
                circuit.x(position)

        circuit.h(3)
        circuit.mcx([0, 1, 2], 3)
        circuit.h(3)

        for position, bit in enumerate(flipped):
            if bit == '0':
                circuit.x(position)

    circuit.name = "Oracle"
    return circuit

 Full Grover Circuit  2 iterations for N=16

In [ ]:
from qiskit.circuit.library.grover_operator import GroverOperator

def run_grover_4(targets, shots=1024):
    oracle    = make_oracle_4(targets)
    grover_op = GroverOperator(oracle)

    search = QuantumCircuit(4, 4)
    search.h(range(4))

    for _ in range(2):
        search.append(grover_op, range(4))

    search.measure(range(4), range(4))

    machine  = AerSimulator()
    compiled = transpile(search, machine)
    result   = machine.run(compiled, shots=shots).result()

    return search, result.get_counts()

### Run and Plot

In [ ]:
marked_states    = ["1111", "0011", "0110", "1110"]
circuit_4, data_4 = run_grover_4(marked_states)

print("Grover 4-Qubit Results:")
print(data_4)

plot_histogram(data_4, title="Grover 4-Qubit - Multiple Targets", figsize=(10, 4))
plt.tight_layout()
plt.show()

### Draw Circuit

In [ ]:
circuit_4.draw('mpl')